# Fraud Monitoring — Publisher-Level Install Analysis

Hourly install patterns, CPI, and retention (D1/D3) by publisher for a given bundle, OS, and geo.

In [ ]:
#@title Colab Authentication

from google.colab import auth
auth.authenticate_user()

In [ ]:
#@title Environment Setup
from google.cloud import bigquery
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 80)

client = bigquery.Client(project='moloco-ods')

def run_query(query, label=''):
    try:
        df = client.query(query).result().to_dataframe()
        status = f'✅ {label}: {len(df)} rows' if len(df) > 0 else f'⚠️ {label}: 0 rows'
        print(status)
        return df
    except Exception as e:
        print(f'❌ {label}: {e}')
        return pd.DataFrame()

## Parameters

In [ ]:
#@title Parameters

MMP_BUNDLE_ID = 'com.netmarble.stonkey'  #@param {type:"string"}
OS = 'ANDROID'  #@param ["ANDROID", "IOS"]
COUNTRY = 'KOR'  #@param {type:"string"}
CAMPAIGN_ID = ''  #@param {type:"string"}
LOOKBACK_DAYS = 7  #@param {type:"integer"}

_campaign_filter = f"AND campaign_id = '{CAMPAIGN_ID}'" if CAMPAIGN_ID else ''
_campaign_filter_cv = f"AND api.campaign.id = '{CAMPAIGN_ID}'" if CAMPAIGN_ID else ''

print(f'Bundle:    {MMP_BUNDLE_ID}')
print(f'OS:        {OS}')
print(f'Country:   {COUNTRY}')
print(f'Campaign:  {CAMPAIGN_ID or "(all)"}')
print(f'Lookback:  {LOOKBACK_DAYS} days')

---

In [ ]:
# placeholder

---
## 1. Daily Publisher Summary (installs, CPI, retention)

In [ ]:
#1. Daily Publisher Summary — installs, CPI, D1/D3 retention

q_pub = f"""
SELECT
  date_utc,
  publisher.app_market_bundle AS publisher_bundle,
  SUM(installs) AS installs,
  SUM(installs_rejected) AS installs_rejected,
  ROUND(SUM(gross_spend_usd), 2) AS spend_usd,
  ROUND(SAFE_DIVIDE(SUM(gross_spend_usd), NULLIF(SUM(installs), 0)), 2) AS cpi,
  SUM(impressions) AS impressions,
  SUM(clicks) AS clicks,
  ROUND(SAFE_DIVIDE(SUM(retained_users_d1), NULLIF(SUM(installs), 0)) * 100, 1) AS retention_d1_pct,
  ROUND(SAFE_DIVIDE(SUM(retained_users_d3), NULLIF(SUM(installs), 0)) * 100, 1) AS retention_d3_pct,
  ROUND(SAFE_DIVIDE(SUM(clicks), NULLIF(SUM(impressions), 0)) * 100, 2) AS ctr_pct
FROM `moloco-ae-view.athena.fact_dsp_publisher`
WHERE date_utc BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL {LOOKBACK_DAYS} DAY) AND CURRENT_DATE()
  AND advertiser.mmp_bundle_id = '{MMP_BUNDLE_ID}'
  AND campaign.country = '{COUNTRY}'
  AND campaign.os = '{OS}'
  {_campaign_filter}
GROUP BY 1, 2
HAVING installs > 0
ORDER BY date_utc DESC, installs DESC
"""

df_pub = run_query(q_pub, '1. Daily Publisher Summary')
if not df_pub.empty:
    numeric_cols = ['installs', 'installs_rejected', 'spend_usd', 'cpi', 'impressions', 'clicks',
                    'retention_d1_pct', 'retention_d3_pct', 'ctr_pct']
    for col in numeric_cols:
        if col in df_pub.columns:
            df_pub[col] = pd.to_numeric(df_pub[col], errors='coerce')

    agg = df_pub.groupby('publisher_bundle').agg(
        total_installs=('installs', 'sum'),
        total_spend=('spend_usd', 'sum'),
        total_impressions=('impressions', 'sum'),
        total_clicks=('clicks', 'sum'),
    ).sort_values('total_installs', ascending=False)
    agg['cpi'] = (agg['total_spend'] / agg['total_installs']).round(2)
    agg['ctr_pct'] = (agg['total_clicks'] / agg['total_impressions'] * 100).round(2)

    print(f'\n  Top publishers by install volume (L{LOOKBACK_DAYS}D):')
    for pub, row in agg.head(15).iterrows():
        print(f'    {pub[:50]:50s}  installs={row["total_installs"]:>6,.0f}  CPI=${row["cpi"]:.2f}  CTR={row["ctr_pct"]:.2f}%')
df_pub

## 2. Hourly Install Patterns by Publisher

In [ ]:
#2. Hourly Install Trend by Publisher (last 24h, from cv postback)

q_hourly = f"""
SELECT
  TIMESTAMP_TRUNC(timestamp, HOUR) AS hour_ts,
  req.app.bundle AS publisher_bundle,
  COUNT(*) AS installs
FROM `focal-elf-631.prod_stream_view.cv`
WHERE timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
  AND UPPER(cv.event) = 'INSTALL'
  AND api.product.app.tracking_bundle = '{MMP_BUNDLE_ID}'
  AND req.device.geo.country = '{COUNTRY}'
  AND req.device.os = '{OS}'
  {_campaign_filter_cv}
GROUP BY 1, 2
ORDER BY 1 ASC, 3 DESC
"""

df_hourly = run_query(q_hourly, '2. Hourly Install Trend (24h)')

if not df_hourly.empty:
    df_hourly['installs'] = pd.to_numeric(df_hourly['installs'], errors='coerce')
    top_pubs = df_hourly.groupby('publisher_bundle')['installs'].sum().sort_values(ascending=False).head(10).index.tolist()
    chart_df = df_hourly[df_hourly['publisher_bundle'].isin(top_pubs)].copy()
    chart_df['hour_ts'] = chart_df['hour_ts'].astype(str)

    HOUR_MS = 3_600_000 * 0.8

    fig = px.bar(chart_df, x='hour_ts', y='installs', color='publisher_bundle',
                 title='Hourly Install Trend by Publisher (Top 10, last 24h)',
                 labels={'hour_ts': 'Time (UTC)', 'installs': 'Installs', 'publisher_bundle': 'Publisher'})
    fig.update_traces(width=HOUR_MS)
    fig.update_layout(barmode='stack', height=500, xaxis=dict(tickformat='%b %d %H:%M', dtick=3_600_000))
    fig.show()

    hourly_total = chart_df.groupby('hour_ts')['installs'].transform('sum')
    chart_df['pct'] = (chart_df['installs'] / hourly_total * 100).round(1)
    fig_pct = px.bar(chart_df, x='hour_ts', y='pct', color='publisher_bundle',
                     title='Hourly Install Share by Publisher (Top 10, last 24h)',
                     labels={'hour_ts': 'Time (UTC)', 'pct': '% of Installs', 'publisher_bundle': 'Publisher'})
    fig_pct.update_traces(width=HOUR_MS)
    fig_pct.update_layout(barmode='stack', height=500,
                          xaxis=dict(tickformat='%b %d %H:%M', dtick=3_600_000),
                          yaxis=dict(range=[0, 100], dtick=20, ticksuffix='%'))
    fig_pct.show()

    import math
    for pub in top_pubs[:5]:
        pub_df = chart_df[chart_df['publisher_bundle'] == pub]
        if pub_df.empty:
            continue
        fig2 = px.bar(pub_df, x='hour_ts', y='installs',
                      title=f'Hourly Trend: {pub[:60]}',
                      labels={'hour_ts': 'Time (UTC)', 'installs': 'Installs'})
        fig2.update_traces(width=HOUR_MS)
        max_val = pub_df['installs'].max()
        y_dtick = max(1, math.ceil(max_val / 6)) if max_val > 0 else 1
        fig2.update_layout(height=300, xaxis=dict(tickformat='%b %d %H:%M', dtick=3_600_000),
                           yaxis=dict(dtick=y_dtick, tick0=0))
        fig2.show()
df_hourly

## 3. D1 / D3 Retention by Publisher

In [ ]:
# 3. D1/D3 Retention by Publisher (fact_dsp_publisher)

q_ret = f"""
SELECT
  publisher.app_market_bundle AS publisher_bundle,
  SUM(installs) AS installs,
  ROUND(SUM(gross_spend_usd), 2) AS spend_usd,
  ROUND(SAFE_DIVIDE(SUM(gross_spend_usd), NULLIF(SUM(installs), 0)), 2) AS cpi,
  ROUND(SUM(retained_users_d1), 1) AS retained_d1,
  ROUND(SUM(retained_users_d3), 1) AS retained_d3,
  ROUND(SAFE_DIVIDE(SUM(retained_users_d1), NULLIF(SUM(installs), 0)) * 100, 1) AS retention_d1_pct,
  ROUND(SAFE_DIVIDE(SUM(retained_users_d3), NULLIF(SUM(installs), 0)) * 100, 1) AS retention_d3_pct
FROM `moloco-ae-view.athena.fact_dsp_publisher`
WHERE date_utc BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL {LOOKBACK_DAYS} DAY) AND DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
  AND advertiser.mmp_bundle_id = '{MMP_BUNDLE_ID}'
  AND campaign.country = '{COUNTRY}'
  AND campaign.os = '{OS}'
  {_campaign_filter}
GROUP BY 1
HAVING installs >= 5
ORDER BY installs DESC
"""

df_ret = run_query(q_ret, '3. D1/D3 Retention by Publisher')
if not df_ret.empty:
    for col in df_ret.columns:
        if col != 'publisher_bundle':
            df_ret[col] = pd.to_numeric(df_ret[col], errors='coerce')
    overall_d1 = (df_ret['retained_d1'].sum() / df_ret['installs'].sum() * 100) if df_ret['installs'].sum() > 0 else 0
    overall_d3 = (df_ret['retained_d3'].sum() / df_ret['installs'].sum() * 100) if df_ret['installs'].sum() > 0 else 0
    print(f'\n  Overall D1 retention: {overall_d1:.1f}%  |  D3 retention: {overall_d3:.1f}%')

    low_d1 = df_ret[(df_ret['retention_d1_pct'] < overall_d1 * 0.5) & (df_ret['installs'] >= 10)]
    if not low_d1.empty:
        print(f'\n  ⚠️ Publishers with D1 retention < {overall_d1 * 0.5:.1f}% (half of avg):')
        for _, row in low_d1.iterrows():
            print(f'    {row["publisher_bundle"][:50]:50s}  installs={row["installs"]:>5,.0f}  D1={row["retention_d1_pct"]:.1f}%  D3={row["retention_d3_pct"]:.1f}%  CPI=${row["cpi"]:.2f}')

    fig = px.scatter(df_ret, x='cpi', y='retention_d1_pct', size='installs',
                     hover_name='publisher_bundle',
                     title=f'Publisher CPI vs D1 Retention (L{LOOKBACK_DAYS}D)',
                     labels={'cpi': 'CPI ($)', 'retention_d1_pct': 'D1 Retention %', 'installs': 'Installs'},
                     hover_data={'retention_d3_pct': ':.1f', 'spend_usd': ':,.2f'})
    fig.update_layout(height=500)
    fig.show()
df_ret

---
## 4. Suspicious Publisher Flags

In [ ]:
# 4. Suspicious Publisher Flags

flags = []

if not df_ret.empty:
    overall_d1 = df_ret['retained_d1'].sum() / df_ret['installs'].sum() * 100 if df_ret['installs'].sum() > 0 else 0
    overall_cpi = df_ret['spend_usd'].sum() / df_ret['installs'].sum() if df_ret['installs'].sum() > 0 else 0

    for _, row in df_ret.iterrows():
        pub = row['publisher_bundle']
        reasons = []

        if row['installs'] >= 10 and row['retention_d1_pct'] < overall_d1 * 0.3:
            reasons.append(f'Very low D1 retention ({row["retention_d1_pct"]:.1f}% vs avg {overall_d1:.1f}%)')
        if row['retention_d1_pct'] == 0 and row['installs'] >= 20:
            reasons.append(f'Zero D1 retention with {row["installs"]:.0f} installs')
        if row['cpi'] < overall_cpi * 0.3 and row['installs'] >= 10:
            reasons.append(f'Suspiciously low CPI (${row["cpi"]:.2f} vs avg ${overall_cpi:.2f})')

        if reasons:
            flags.append({'publisher': pub, 'installs': row['installs'], 'cpi': row['cpi'],
                          'd1_ret': row['retention_d1_pct'], 'reasons': reasons})

if not df_hourly.empty:
    pub_hour_agg = df_hourly.groupby('publisher_bundle').agg(
        total=('installs', 'sum'), hours_active=('hour_ts', 'nunique')).reset_index()
    for _, row in pub_hour_agg.iterrows():
        if row['total'] >= 20 and row['hours_active'] <= 2:
            existing = next((f for f in flags if f['publisher'] == row['publisher_bundle']), None)
            reason = f'Installs concentrated in {row["hours_active"]} hour(s) only'
            if existing:
                existing['reasons'].append(reason)
            else:
                flags.append({'publisher': row['publisher_bundle'], 'installs': row['total'],
                              'cpi': None, 'd1_ret': None, 'reasons': [reason]})

print(f'{"=" * 60}')
print(f'  Fraud Monitoring Summary')
print(f'  {MMP_BUNDLE_ID} / {OS} / {COUNTRY} / L{LOOKBACK_DAYS}D')
print(f'{"=" * 60}')

print('\n  Flagging Criteria:')
print('    1. D1 retention < 30% of overall avg  (min 10 installs)')
print('    2. Zero D1 retention                  (min 20 installs)')
print('    3. CPI < 30% of overall avg           (min 10 installs)')
print('    4. Installs in ≤2 hours over 24h      (min 20 installs)')

if flags:
    print(f'\n  ⚠️ {len(flags)} publisher(s) flagged:\n')
    for f in sorted(flags, key=lambda x: x['installs'], reverse=True):
        print(f'  Publisher: {f["publisher"]}')
        print(f'    Installs: {f["installs"]:.0f}  |  CPI: {"$" + str(f["cpi"]) if f["cpi"] else "N/A"}  |  D1 ret: {str(f["d1_ret"]) + "%" if f["d1_ret"] is not None else "N/A"}')
        for r in f['reasons']:
            print(f'    🚩 {r}')
        print()
else:
    print('\n  ✅ No suspicious publishers flagged')